# Базовый эксперимент с моделью SIR

**Цель:** Запустить один эксперимент с фиксированными параметрами (по умолчанию)
и сохранить динамику численности агентов. Это служит для проверки работоспособности
модели и получения базового понимания эпидемического процесса.

## Инициализация проекта и загрузка пакетов

In [ ]:
using DrWatson
@quickactivate "project"
using Agents, DataFrames, Plots
using JLD2

Подключение модуля с определением модели SIR и вспомогательных функций

In [ ]:
include(srcdir("sir_model.jl"))

## Параметры эксперимента

Определяем параметры модели в виде словаря для удобного управления.
Все параметры имеют следующие значения:

- `Ns` — численность населения в трёх городах (по 1000 человек)
- `β_und` — интенсивность заражения невыявленными больными (0.5)
- `β_det` — интенсивность заражения выявленными больными (0.05)
- `infection_period` — длительность болезни (14 дней)
- `detection_time` — время до выявления после заражения (7 дней)
- `death_rate` — вероятность смерти при завершении болезни (2%)
- `reinfection_probability` — вероятность повторного заражения (10%)
- `Is` — начальное количество инфицированных в каждом городе (только в третьем)
- `seed` — зерно генератора случайных чисел (для воспроизводимости)
- `n_steps` — количество дней симуляции (100 дней)

In [ ]:
params = Dict(
    :Ns => [1000, 1000, 1000],
    :β_und => [0.5, 0.5, 0.5],
    :β_det => [0.05, 0.05, 0.05],
    :infection_period => 14,
    :detection_time => 7,
    :death_rate => 0.02,
    :reinfection_probability => 0.1,
    :Is => [0, 0, 1],
    :seed => 42,
    :n_steps => 100,
)

## Инициализация модели

Создаём модель SIR с заданными параметрами. Функция `initialize_sir`
создаёт агентов, распределяет их по городам и задаёт начальные условия.

In [ ]:
model = initialize_sir(; params...)

## Подготовка массивов для хранения данных

Создаём массивы для записи динамики численности популяции на каждом шаге:
- `times` — номер шага (дня)
- `S_vals` — количество восприимчивых (Susceptible)
- `I_vals` — количество инфицированных (Infectious)
- `R_vals` — количество выздоровевших (Recovered)
- `total_vals` — общая численность популяции (с учётом умерших)

In [ ]:
times = Int[]
S_vals = Int[]
I_vals = Int[]
R_vals = Int[]
total_vals = Int[]

## Запуск симуляции

Выполняем симуляцию вручную на `n_steps` шагов. На каждом шаге:
1. Применяем шаг моделирования `Agents.step!`
2. Сохраняем текущие значения численностей популяций

Используются вспомогательные функции, определённые в `sir_model.jl`:
- `susceptible_count(model)` — количество восприимчивых
- `infected_count(model)` — количество инфицированных
- `recovered_count(model)` — количество выздоровевших
- `total_count(model)` — общая численность (живые агенты)

In [ ]:
for step = 1:params[:n_steps]

Выполняем один шаг моделирования

In [ ]:
    Agents.step!(model, 1)

Сохраняем данные

In [ ]:
    push!(times, step)
    push!(S_vals, susceptible_count(model))
    push!(I_vals, infected_count(model))
    push!(R_vals, recovered_count(model))
    push!(total_vals, total_count(model))
end

## Создание DataFrame для анализа

Преобразуем массивы в DataFrame для удобной работы с данными:
- `agent_df` — данные по группам S, I, R
- `model_df` — данные по общей численности

In [ ]:
agent_df = DataFrame(
    time = times,
    susceptible = S_vals,
    infected = I_vals,
    recovered = R_vals
)

model_df = DataFrame(
    time = times,
    total = total_vals
)

## Визуализация результатов

Строим график динамики эпидемии, который показывает:
- Кривую восприимчивых (S) — убывает по мере распространения инфекции
- Кривую инфицированных (I) — возрастает, достигает пика, затем убывает
- Кривую выздоровевших (R) — возрастает и выходит на плато
- Общую численность популяции (пунктирная линия) — уменьшается за счёт смертности

In [ ]:
plot(
    agent_df.time,
    agent_df.susceptible,
    label = "Восприимчивые",
    xlabel = "Дни",
    ylabel = "Количество",
)

plot!(agent_df.time, agent_df.infected, label = "Инфицированные")
plot!(agent_df.time, agent_df.recovered, label = "Выздоровевшие")
plot!(agent_df.time, model_df.total, label = "Всего (включая умерших)", linestyle = :dash)

Сохраняем график в каталог `plots/`

In [ ]:
savefig(plotsdir("sir_basic_dynamics.png"))

## Сохранение данных

Сохраняем DataFrames в JLD2-файлы для последующего анализа.
Используем `@save` макрос из пакета JLD2 для удобного сохранения.

In [ ]:
@save datadir("sir_basic_agent.jld2") agent_df
@save datadir("sir_basic_model.jld2") model_df

## Анализ результатов

На полученном графике можно визуально оценить:

1. **Пик эпидемии** — максимальное значение кривой I(t)
2. **Скорость распространения** — крутизна роста кривой I(t)
3. **Влияние смертности** — разница между общей численностью и суммой S+I+R
4. **Итоговый охват** — значение кривой R(t) на плато

Этот базовый эксперимент служит отправной точкой для более сложных
исследований, таких как параметрическое сканирование и оптимизация.